# 🏹 Juegos del Hambre de Modelos Fundacionales (Servidor Colab Pro)
Este notebook levanta el servidor central para **12 estudiantes** usando una GPU **NVIDIA L4 (24 GB)** o **A100 (40 GB)**.

- **3 Modelos soportados:** `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct`
- **Presupuesto máximo por estudiante:** `1.0 PFLOP` (`100 créditos`)
- **Exposición pública automática:** Genera un enlace HTTPS con `cloudflared` para pegarlo en `config.py` de los estudiantes.

In [ ]:
# 1. Verificar GPU asignada (Recomendado: L4 24GB o A100 40GB) y montar Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARENA_SAVE_DIR'] = '/content/drive/MyDrive/hunger_games_fm_arena'
os.makedirs(os.environ['ARENA_SAVE_DIR'], exist_ok=True)
print('Guardando estado en:', os.environ['ARENA_SAVE_DIR'])

In [ ]:
# 2. Instalar dependencias y túnel Cloudflared
!pip install -q fastapi uvicorn transformers peft accelerate pycloudflared

In [ ]:
# 3. Iniciar servidor FastAPI en segundo plano y abrir URL pública para los 12 equipos
import threading, time, uvicorn
from pycloudflared import try_cloudflare
import server  # Asegúrate de subir server.py a la sesión de Colab o en la carpeta de Drive

os.environ['RUN_REAL_GPU'] = '1'
threading.Thread(target=lambda: uvicorn.run(server.app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(2)
tunnel = try_cloudflare(port=8000)
print('=' * 70)
print('🌍 COMPARTE ESTA URL CON TUS 12 ESTUDIANTES (COLAB_API_URL en config.py):')
print(tunnel.tunnel)
print('=' * 70)

In [ ]:
# 4. Consultar cuántos equipos ya entrenaron y disparar la Arena PvP Final
import urllib.request, json

with urllib.request.urlopen('http://127.0.0.1:8000/leaderboard') as r:
    lb = json.loads(r.read().decode())
print(f"Tributos registrados: {len(lb['tributos'])} / 12")
for t in lb['tributos']:
    print(f" - {t['equipo']} ({t['modelo_base'].split('/')[-1]}): Radar medio={t['media_radar']} | Créditos={t['creditos_usados']}")

# Ejecutar la Arena PvP y guardar arena_pvp_replay.json para cargarlo en arena_pixelart.html
req = urllib.request.Request('http://127.0.0.1:8000/correr_arena_pvp', method='POST')
with urllib.request.urlopen(req) as r:
    replay = json.loads(r.read().decode())
print(f"\n🏆 GANADOR DE LA ARENA PVP: {replay['ganador']} en {replay['turnos_totales']} turnos")